# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [2]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [6]:
# TODO: implementa la lectura del CSV.
with DATA_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    row = rows[0] if rows else {}

print(f"Número de filas: {len(rows)}")
print(f"Columnas detectadas: {list(row.keys())}")

Número de filas: 5
Columnas detectadas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? El campo sample_id
2. ¿Cuántas características numéricas hay? 11
3. ¿Qué columna extra añadirías para probar el contrato? un columna cualquiera con un nombre random
4. ¿Qué valor de `ph` sería claramente inválido? uno fuera del rango

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [9]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    # TODO: declara las 11 características con Field(ge=..., le=...).
    model_config=ConfigDict(extra="forbid")
    
    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

# TODO: separa sample_id de la fila y valida solo las características del modelo.
model_row = {k: v for k, v in row.items() if k != "sample_id"}
request = WineQualityRequestPydantic.model_validate(model_row)

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.
- Una fila válida: NINGUN ERROR 
- Una columna `unexpected_field`: GRACIAS AL MODEL_CONFIG = CONFIGDICT(EXTRA="FORBID") SALDRA ValidationError
- Un `ph` fuera de rango: GRACIAS AL Field(le=4.5) SALDRA ValidationError

In [11]:
# TODO: completa los casos y captura ValidationError.
# try:
#     ...
# except ValidationError as error:
#     print(error)

from pydantic import ValidationError

# 1. Creamos un diccionario base que sabemos que es matemáticamente perfecto
valid_data = {
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.7,
    "citric_acid": 0.0,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "ph": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4
}

print("--- CASO 1: Fila Válida ---")
try:
    # Usamos **valid_data para desempaquetar el diccionario como argumentos
    request = WineQualityRequestPydantic(**valid_data)
    print("Éxito: La fila válida pasó el contrato sin problemas.")
except ValidationError as error:
    print(f"Fallo inesperado: {error}")


print("\n--- CASO 2: Columna inesperada ---")
# Hacemos una copia e inyectamos un campo que no existe en el contrato
invalid_data_extra = valid_data.copy()
invalid_data_extra["unexpected_field"] = "dato corrupto"

try:
    WineQualityRequestPydantic(**invalid_data_extra)
    print("¡PELIGRO CRÍTICO! El contrato dejó pasar una columna extra.")
except ValidationError as error:
    print("Rechazo correcto (Columna extra bloqueada):")
    print(error)


print("\n--- CASO 3: pH fuera de rango ---")
# Hacemos una copia e inyectamos un valor físicamente imposible
invalid_data_ph = valid_data.copy()
invalid_data_ph["ph"] = 99.0

try:
    WineQualityRequestPydantic(**invalid_data_ph)
    print("¡PELIGRO CRÍTICO! El contrato dejó pasar un pH imposible.")
except ValidationError as error:
    print("Rechazo correcto (pH fuera de rango bloqueado):")
    print(error)


--- CASO 1: Fila Válida ---
Éxito: La fila válida pasó el contrato sin problemas.

--- CASO 2: Columna inesperada ---
Rechazo correcto (Columna extra bloqueada):
1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='dato corrupto', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden

--- CASO 3: pH fuera de rango ---
Rechazo correcto (pH fuera de rango bloqueado):
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=99.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [14]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float
    
# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.
instancia_nativa = WineQualityRequestDataclass(
    fixed_acidity=7.4,
    volatile_acidity=0.7,
    citric_acid=0.0,
    residual_sugar=1.9,
    chlorides=0.076,
    free_sulfur_dioxide=11.0,
    total_sulfur_dioxide=34.0,
    density=0.9978,
    ph=99.0,          # Dato físicamente imposible
    sulphates=0.56,
    alcohol="mucho"   # Tipo de dato incorrecto
)

print("--- RESULTADO DEL DATACLASS NATIVO ---")
print("Instancia creada sin errores:")
print(instancia_nativa)
print(f"Valor del pH guardado: {instancia_nativa.ph}")
print(f"Tipo de dato guardado en alcohol: {type(instancia_nativa.alcohol)}")

--- RESULTADO DEL DATACLASS NATIVO ---
Instancia creada sin errores:
WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=99.0, sulphates=0.56, alcohol='mucho')
Valor del pH guardado: 99.0
Tipo de dato guardado en alcohol: <class 'str'>


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en _el archivo contract_________ porque __valida los datos en tiempo de ejecucion ________. Usaría `dataclass` en _en el trafico interno de la aplicacion _________ porque __es mas ligero________.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [15]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
)

# TODO: construye vector usando getattr(request, name).
# vector = [...]
# Construimos el vector posicional dinámicamente
vector = [getattr(request, name) for name in FEATURE_NAMES]

print(f"Vector generado: {vector}")

Vector generado: [7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

si intercambias la posición de la densidad y el alcohol en esta tupla, el vector resultante seguirá teniendo 11 valores numéricos perfectos. El clasificador no lanzará ninguna alerta y ejecutará la inferencia, pero aplicará los pesos matemáticos del alcohol a la densidad, devolviendo un resultado inválido

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [16]:
from pydantic import BaseModel, Field

class WineQualityPredictionPydantic(BaseModel):
    # ¿Qué muestra se ha procesado?
    sample_id: str = Field(min_length=1, description="Identificador original de la muestra")
    
    # ¿Qué categoría se ha predicho?
    quality_band: str = Field(min_length=1, description="Clasificación final (ej. needs_review, acceptable)")
    
    # ¿Con qué confianza?
    confidence: float = Field(ge=0.0, le=1.0, description="Probabilidad matemática del clasificador")
    
    # ¿Qué versiones exactas se utilizaron?
    model_version: str = Field(min_length=1, description="Versión del artefacto .joblib")
    preprocessing_version: str = Field(min_length=1, description="Versión del código de transformación")

# TODO: escribe un ejemplo de salida para red-001.
prediccion_ejemplo = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="needs_review",
    confidence=0.77,
    model_version="wine-quality-rf-v1",
    preprocessing_version="wine-red-features-v1"
)

print("--- CONTRATO DE SALIDA GENERADO ---")
print(prediccion_ejemplo.model_dump_json(indent=2))

--- CONTRATO DE SALIDA GENERADO ---
{
  "sample_id": "red-001",
  "quality_band": "needs_review",
  "confidence": 0.77,
  "model_version": "wine-quality-rf-v1",
  "preprocessing_version": "wine-red-features-v1"
}


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.